# Building a Full ML Pipeline — Titanic

This notebook is the culmination of the earlier scaling/encoding/imputation notebooks: instead of doing each preprocessing step manually and gluing the results together with `np.concatenate()` (as in `titanic-without-using-pipeline.ipynb`), every step — imputation, encoding, scaling, feature selection, and the model itself — is chained into a single `Pipeline` object. One `.fit()`, one `.predict()`, and (crucially) one object to save and reload later.

## 1. Imports and Setup

In [1]:
import numpy as np
import pandas as pd

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline,make_pipeline
from sklearn.feature_selection import SelectKBest,chi2
from sklearn.tree import DecisionTreeClassifier

In [3]:
df = pd.read_csv('/kaggle/input/datasets/muhammadzafranai/titanics-dataset/train.csv')

In [4]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


# Let's Plan

### The Plan

Five steps will be chained together, in order:
1. **Impute** missing `Age` and `Embarked` values
2. **One-hot encode** `Sex` and `Embarked`
3. **Scale** everything to 0–1 with `MinMaxScaler`
4. **Select** the 8 most useful features out of the resulting columns
5. **Train** a `DecisionTreeClassifier` on what's left

In [5]:
df.drop(columns=['PassengerId','Name','Ticket','Cabin'],inplace=True)

In [6]:
# Step 1 -> train/test/split
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['Survived']),
                                                 df['Survived'],
                                                 test_size=0.2,
                                                random_state=42)

In [7]:
X_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
331,1,male,45.5,0,0,28.5000,S
733,2,male,23.0,0,0,13.0000,S
382,3,male,32.0,0,0,7.9250,S
704,3,male,26.0,1,0,7.8542,S
813,3,female,6.0,4,2,31.2750,S


In [8]:
y_train.sample(5)

780    1
19     1
861    0
626    0
75     0
Name: Survived, dtype: int64

## 2. Step 1 — Imputation (`trf1`)

`Age` (column index 2) gets the default `strategy='mean'`; `Embarked` (index 6) gets `strategy='most_frequent'`, which makes more sense for a categorical column than a mean would. `remainder='passthrough'` keeps every other column untouched.

In [9]:
# imputation transformer
trf1 = ColumnTransformer([
    ('impute_age',SimpleImputer(),[2]),
    ('impute_embarked',SimpleImputer(strategy='most_frequent'),[6])
],remainder='passthrough')


## 3. Step 2 — One-Hot Encoding (`trf2`)

`Sex` and `Embarked` (indices 1 and 6 — note these indices refer to the *original* column positions, since each `ColumnTransformer` step here is applied independently to the original DataFrame layout, not chained automatically) get one-hot encoded together in a single call.

### ⚠️ `sparse=False` Is Removed in Scikit-Learn 1.4+

Same issue flagged in the earlier encoding notebooks — renamed to `sparse_output`.

In [10]:
# one-hot encoding
trf2 = ColumnTransformer([
    ('ohe_sex_embarked', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), [1, 6])
], remainder='passthrough')


In [11]:

# Scaling
trf3 = ColumnTransformer([
    ('scale',MinMaxScaler(),slice(0,10))
])


## 4. Step 3 — Scaling (`trf3`)

By this point, `trf1` and `trf2` have already run (Age/Embarked imputed, Sex/Embarked one-hot encoded), so the data now has **10 columns** total: 5 from one-hot encoding (2 for Sex + 3 for Embarked) plus the 5 remaining original columns (Pclass, Age, SibSp, Parch, Fare). `slice(0, 10)` selects all of them for scaling — the whole point of chaining these into a `Pipeline` (next section) is that each step automatically receives the *previous* step's output, so this column count lines up correctly without you having to track it by hand.

In [12]:
# Feature selection
trf4 = SelectKBest(score_func=chi2,k=8)

## 5. Step 4 — Feature Selection (`trf4`)

`SelectKBest` keeps only the `k` features most strongly associated with the target, dropping the rest. `score_func=chi2` measures that association for **categorical/discrete-style** targets — but chi2 has a hard requirement worth knowing: **it needs every value to be non-negative**. That's exactly why `MinMaxScaler` (which forces everything into the `[0, 1]` range) ran immediately before this step — chi2 would raise an error on raw, possibly-negative data.

In [13]:
# train the model
trf5 = DecisionTreeClassifier()

# Create Pipeline

In [14]:
pipe = Pipeline([
    ('trf1',trf1),
    ('trf2',trf2),
    ('trf3',trf3),
    ('trf4',trf4),
    ('trf5',trf5)
])

# Pipeline Vs make_pipeline

Pipeline requires naming of steps, make_pipeline does not.

(Same applies to ColumnTransformer vs make_column_transformer)

In [15]:
# Alternate Syntax
pipe = make_pipeline(trf1,trf2,trf3,trf4,trf5)

In [16]:
# train
pipe.fit(X_train,y_train)

Pipeline(steps=[('columntransformer-1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('impute_age', SimpleImputer(),
                                                  [2]),
                                                 ('impute_embarked',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  [6])])),
                ('columntransformer-2',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ohe_sex_embarked',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  [1, 6])])),
                ('columntransformer-3',
                 ColumnTransformer(transformers=[('scale', MinMaxScaler(),
                                                  slice(0, 10, None))])),
                ('selectkbest',
                 SelectKBest(k=8,
                             score_func=<function chi2 at 0x7de7adc57c40>)),
                ('decisiontreeclassifier', DecisionTreeClassifier())])

`.fit()` on a `Pipeline` runs every step in order automatically — `trf1.fit_transform()`, then `trf2.fit_transform()` on *that* output, and so on down to the classifier — all from this one call.

# Explore the Pipeline

In [17]:
# Code here
pipe.named_steps

{'columntransformer-1': ColumnTransformer(remainder='passthrough',
                   transformers=[('impute_age', SimpleImputer(), [2]),
                                 ('impute_embarked',
                                  SimpleImputer(strategy='most_frequent'),
                                  [6])]),
 'columntransformer-2': ColumnTransformer(remainder='passthrough',
                   transformers=[('ohe_sex_embarked',
                                  OneHotEncoder(handle_unknown='ignore',
                                                sparse_output=False),
                                  [1, 6])]),
 'columntransformer-3': ColumnTransformer(transformers=[('scale', MinMaxScaler(), slice(0, 10, None))]),
 'selectkbest': SelectKBest(k=8, score_func=<function chi2 at 0x7de7adc57c40>),
 'decisiontreeclassifier': DecisionTreeClassifier()}

In [18]:
# Code here
pipe.named_steps['trf1'].transformers_[0][1].statistics_

KeyError: 'trf1'

`.named_steps` gives you back each individual fitted transformer/model by name — useful for inspecting any single step's learned state after the fact (as done further down for feature selection).

In [ ]:
# Display Pipeline

from sklearn import set_config
set_config(display='diagram')

### ⚠️ Missing: Actually Displaying the Diagram

`set_config(display='diagram')` only changes a *setting* — it doesn't show anything by itself. To see the visual diagram, evaluate the pipeline object itself as the last line of a cell (Jupyter renders it as rich HTML).

In [ ]:
pipe   # renders as an interactive diagram, not just text, now that display='diagram' is set


In [ ]:
# Predict
y_pred = pipe.predict(X_test)

In [ ]:
y_pred

In [ ]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

### Looking Beyond Accuracy — A Confusion Matrix

A single accuracy number hides *how* the model is wrong — e.g. whether it's missing survivors or wrongly flagging non-survivors. A confusion matrix breaks that down.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.title('Confusion Matrix — Titanic Survival')
plt.show()


### Which Features Actually Got Selected?

`SelectKBest` chose the best 8 out of 10 columns, but nothing so far shows *which* 2 were dropped. `get_support()` reveals exactly that.

In [ ]:
selected = pipe.named_steps['trf4'].get_support()  # step name from make_pipeline's auto-naming
# alternatively, if using the named `Pipeline` from earlier: pipe.named_steps['trf4'].get_support()
print('Features kept (boolean mask):', selected)


# Cross Validation using Pipeline

In [ ]:
# cross validation using cross_val_score
from sklearn.model_selection import cross_val_score
cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy').mean()

A single `train_test_split()` accuracy score can vary a lot depending on which rows happened to land in the test set. `cross_val_score` runs the whole pipeline 5 times on 5 different train/test splits and averages the result — a much more reliable estimate of how the model actually performs.

# GridSearch using Pipeline

In [ ]:
# gridsearchcv
params = {
    'trf5__max_depth':[1,2,3,4,5,None]
}

### The `stepname__paramname` Naming Convention

`'trf5__max_depth'` looks unusual, but it's how `GridSearchCV` knows *which step's* hyperparameter to tune inside a multi-step `Pipeline` — `trf5` is the step name given when the `Pipeline` was created, and `max_depth` is a parameter of `DecisionTreeClassifier` specifically. The double underscore (`__`) is the required separator. (Note: this only works with the **named** `Pipeline` from earlier, since `make_pipeline` auto-generates step names like `decisiontreeclassifier` instead of `trf5` — worth keeping in mind if switching between the two syntaxes.)

In [ ]:
from sklearn.model_selection import GridSearchCV
grid = GridSearchCV(pipe, params, cv=5, scoring='accuracy')
grid.fit(X_train, y_train)

In [ ]:
grid.best_score_

In [ ]:
grid.best_params_

### ⚠️ Important: The Exported Model Should Be the *Tuned* One

`grid.fit()` doesn't just find the best hyperparameters — by default (`refit=True`), it also **automatically retrains** a final pipeline using those best parameters on the *entire* training set. That fully-tuned pipeline is available as `grid.best_estimator_`. Exporting the original `pipe` (below, in the original notebook) would throw away everything the grid search just found, and ship the untuned, default-`max_depth` model instead — worth double-checking which object actually gets pickled.

# Exporting the Pipeline

In [ ]:
# export the TUNED pipeline (found by GridSearchCV), not the original untuned `pipe`
import pickle
pickle.dump(grid.best_estimator_, open('pipe.pkl', 'wb'))


## Summary

| Task | Method |
|---|---|
| Chain preprocessing + model into one object | `Pipeline([(name, step), ...])` or `make_pipeline(step, ...)` |
| See a step's fitted internals | `pipe.named_steps['name']` |
| Visualize the pipeline structure | `set_config(display='diagram')`, then evaluate `pipe` alone |
| More reliable accuracy estimate | `cross_val_score(pipe, X, y, cv=5)` |
| Tune a step's hyperparameter | `GridSearchCV` with `'stepname__paramname'` keys |
| Get the actually-tuned model after grid search | `grid.best_estimator_` *(not the original `pipe`)* |
| See which features survived selection | `selectkbest_step.get_support()` |
| Save the whole pipeline as one file | `pickle.dump(grid.best_estimator_, open('pipe.pkl', 'wb'))` |

### Why this beats the manual approach
One object (`pipe.pkl`) captures every fitted transformer *and* the model, in the exact order they need to run. No risk of forgetting a step, mismatching column order, or accidentally re-fitting on test data — the entire failure mode explored in `titanic-without-using-pipeline.ipynb` and `day28.ipynb` simply can't happen here.